# Human-in-the-Loop Adjudication with Step Functions (OPTIONAL)

This notebook is **optional** — no other notebooks depend on it.

In the end-to-end orchestration notebook (NB 03/02), the Intake Orchestrator auto-approved a clean claim. But what happens when a claim has **flags** — an exclusion triggered, low identity confidence, or cross-document inconsistencies?

The supervisor can’t auto-approve a flagged claim. It needs a **human adjudicator** to review the evidence and make a decision. This notebook demonstrates that pattern using **AWS Step Functions** with the **task token callback** mechanism:

1. The supervisor runs pre-processing and detects flags in the verification output
2. Instead of auto-approving, it starts a Step Functions execution that **pauses and waits** for a human decision
3. You (the workshop attendee) act as the adjudicator — review the claim and submit a decision
4. Step Functions resumes, updates DynamoDB, and the pipeline completes

### Why Step Functions?

The callback pattern is ideal for human-in-the-loop workflows because:
- The state machine **pauses without consuming compute** — it can wait hours or days
- The task token is a secure, one-time credential for resuming the workflow
- In production, the human review UI could be a web app, email link, or Amazon Quick Automate task queue — the callback pattern works the same regardless

```
Supervisor → detects flags → starts Step Functions execution
                                    │
                              [pauses, waits for callback]
                                    │
                          Human reviews claim in notebook
                                    │
                          send_task_success(token, decision)
                                    │
                          Step Functions resumes → updates DynamoDB
```

## Part A — Setup

In [ ]:
import boto3
import json
import sys
import os
import re
import time
import uuid
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal

from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

# ── Path setup ──────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
REPO_ROOT = _cwd
for _parent in [_cwd] + list(_cwd.parents):
    if (_parent / "agents").is_dir() and (_parent / "mcp_servers").is_dir():
        REPO_ROOT = _parent
        break
sys.path.insert(0, str(REPO_ROOT))

MCP_SERVER_PATH = REPO_ROOT / "mcp_servers" / "socotra_mock"
MOCK_DATA_PATH = MCP_SERVER_PATH / "mock_data.json"

from agents import authenticator_agent
from agents import extractor_agent
from agents import policy_verification_agent

REGION = "us-east-1"
S3_BUCKET = os.environ.get("WORKSHOP_S3_BUCKET", "")
if not S3_BUCKET:
    # Auto-detect: find the workshop bucket by naming convention
    for _b in boto3.client("s3", region_name=REGION).list_buckets().get("Buckets", []):
        if _b["Name"].startswith("agentic-workshop-"):
            S3_BUCKET = _b["Name"]
            break
assert S3_BUCKET, "Could not find workshop S3 bucket. Set WORKSHOP_S3_BUCKET env var or ensure the CloudFormation stack is deployed."
S3_PREFIX = "claims-processing/claimant-data/"
DYNAMODB_TABLE = "claims-metadata"

s3_client = boto3.client("s3", region_name=REGION)
dynamodb = boto3.resource("dynamodb", region_name=REGION)
sfn_client = boto3.client("stepfunctions", region_name=REGION)
iam_client = boto3.client("iam", region_name=REGION)
sts_client = boto3.client("sts", region_name=REGION)

ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

print("✅ Setup complete")
print(f"   Account   : {ACCOUNT_ID}")
print(f"   Region    : {REGION}")

## Part B — Create the Step Functions State Machine

The state machine has one key step: a **Task** state with `.waitForTaskToken`. This state pauses the execution and generates a unique task token. The execution won’t resume until an external process calls `SendTaskSuccess` with that token.

We store the task token in DynamoDB alongside the claim data so the adjudicator cell can retrieve it later.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Create IAM role for Step Functions and Activity for human review
# ─────────────────────────────────────────────────────────────────────────────

SFN_ROLE_NAME = "claims-hitl-sfn-role"
STATE_MACHINE_NAME = "claims-hitl-adjudication"
ACTIVITY_NAME = "claims-human-review"

trust_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "states.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
})

sfn_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["dynamodb:PutItem", "dynamodb:UpdateItem", "dynamodb:GetItem"],
        "Resource": f"arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{DYNAMODB_TABLE}"
    }]
})

try:
    role_response = iam_client.create_role(
        RoleName=SFN_ROLE_NAME,
        AssumeRolePolicyDocument=trust_policy,
        Description="Role for claims HITL Step Functions state machine"
    )
    SFN_ROLE_ARN = role_response["Role"]["Arn"]
    iam_client.put_role_policy(
        RoleName=SFN_ROLE_NAME,
        PolicyName="claims-hitl-dynamodb-access",
        PolicyDocument=sfn_policy
    )
    print(f"✅ Created IAM role: {SFN_ROLE_NAME}")
    time.sleep(10)
except iam_client.exceptions.EntityAlreadyExistsException:
    SFN_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{SFN_ROLE_NAME}"
    print(f"ℹ️  IAM role already exists: {SFN_ROLE_NAME}")

# Create an Activity (the human review task)
try:
    activity_resp = sfn_client.create_activity(name=ACTIVITY_NAME)
    ACTIVITY_ARN = activity_resp["activityArn"]
    print(f"✅ Created activity: {ACTIVITY_NAME}")
except Exception as e:
    ACTIVITY_ARN = f"arn:aws:states:{REGION}:{ACCOUNT_ID}:activity:{ACTIVITY_NAME}"
    print(f"ℹ️  Activity may already exist: {ACTIVITY_NAME}")

print(f"   Role ARN: {SFN_ROLE_ARN}")
print(f"   Activity ARN: {ACTIVITY_ARN}")



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Create the Step Functions state machine
#
# Uses an Activity-based wait pattern. The state machine pauses at the
# WaitForHumanDecision state until send_task_success is called.
# ─────────────────────────────────────────────────────────────────────────────

state_machine_definition = json.dumps({
    "Comment": "Claims HITL Adjudication - waits for human decision via Activity",
    "StartAt": "WaitForHumanDecision",
    "States": {
        "WaitForHumanDecision": {
            "Type": "Task",
            "Resource": ACTIVITY_ARN,
            "TimeoutSeconds": 86400,
            "ResultPath": "$.human_decision",
            "End": True
        }
    }
})

import time as _time

for _attempt in range(6):
    try:
        sm_response = sfn_client.create_state_machine(
            name=STATE_MACHINE_NAME,
            definition=state_machine_definition,
            roleArn=SFN_ROLE_ARN,
            type="STANDARD",
        )
        STATE_MACHINE_ARN = sm_response["stateMachineArn"]
        print(f"\u2705 Created state machine: {STATE_MACHINE_NAME}")
        break
    except sfn_client.exceptions.StateMachineAlreadyExists:
        STATE_MACHINE_ARN = f"arn:aws:states:{REGION}:{ACCOUNT_ID}:stateMachine:{STATE_MACHINE_NAME}"
        print(f"\u2139\ufe0f  State machine already exists: {STATE_MACHINE_NAME}")
        break
    except sfn_client.exceptions.StateMachineDeleting:
        print(f"   State machine is being deleted, waiting 10s... (attempt {_attempt + 1}/6)")
        _time.sleep(10)
    except Exception as e:
        if "StateMachineDeleting" in str(e):
            print(f"   State machine is being deleted, waiting 10s... (attempt {_attempt + 1}/6)")
            _time.sleep(10)
        else:
            raise

print(f"   ARN: {STATE_MACHINE_ARN}")



## Part C — Build the Supervisor with HITL-Aware Adjudication

The supervisor has the same tools as before, but `adjudicate_claim` now has two paths:
- **Auto-approve** if all checks pass (happy path)
- **Escalate to Step Functions** if flags are detected — starts the state machine and returns `ESCALATED`

We use a **flagged claim** for this notebook: the death circumstances mention suicide, which triggers the `suicide_within_2_years` exclusion in the Socotra mock data.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 1: Retrieve claim documents from S3
# ─────────────────────────────────────────────────────────────────────────────

LOCAL_DOCS_DIR = Path("claim_documents")
LOCAL_DOCS_DIR.mkdir(exist_ok=True)

@tool
def retrieve_claim_documents(s3_bucket: str, s3_prefix: str) -> str:
    """Download claim PDF documents from S3.
    Args:
        s3_bucket: S3 bucket name
        s3_prefix: S3 key prefix
    """
    response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix)
    s3_objects = response.get("Contents", [])
    downloaded = []
    for obj in s3_objects:
        key = obj["Key"]
        filename = key.split("/")[-1]
        if not filename or not filename.endswith(".pdf"):
            continue
        local_path = LOCAL_DOCS_DIR / filename
        s3_client.download_file(s3_bucket, key, str(local_path))
        downloaded.append(str(local_path))
    return json.dumps({"status": "success", "document_paths": downloaded})


# ─────────────────────────────────────────────────────────────────────────────
# Tool 2: Pre-processing graph
# ─────────────────────────────────────────────────────────────────────────────

SOCOTRA_SERVER_SCRIPT = str(MCP_SERVER_PATH / "server.py")
socotra_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command=sys.executable,
            args=[SOCOTRA_SERVER_SCRIPT],
            env={"MOCK_DATA_PATH": str(MOCK_DATA_PATH), **os.environ}
        )
    )
)

auth_passed = True

@tool
def run_preprocessing_graph(claim_prompt: str) -> str:
    """Run the 3-node pre-processing graph.
    Args:
        claim_prompt: Full claim submission prompt
    """
    global auth_passed
    auth_passed = True
    mcp_tools = socotra_mcp_client.list_tools_sync()
    auth_agent = authenticator_agent.build_agent(mcp_tools)
    output_dir = Path("extracted_output")
    output_dir.mkdir(exist_ok=True)
    extract_agent = extractor_agent.build_agent(output_dir=output_dir)
    verify_agent = policy_verification_agent.build_agent(mcp_tools)
    def check_auth_passed(state): return auth_passed
    builder = GraphBuilder()
    builder.add_node(auth_agent, "authenticate")
    builder.add_node(extract_agent, "extract")
    builder.add_node(verify_agent, "verify_policy")
    builder.set_entry_point("authenticate")
    builder.add_edge("authenticate", "extract", condition=check_auth_passed)
    builder.add_edge("extract", "verify_policy")
    graph = builder.build()
    return str(graph(claim_prompt))


print("✅ retrieve_claim_documents and run_preprocessing_graph tools defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 3: HITL-aware adjudication
#
# If flags detected → start Step Functions execution (pauses for human)
# If clean → auto-approve
# ─────────────────────────────────────────────────────────────────────────────

@tool
def adjudicate_claim(claim_id: str, verification_summary: str, auth_summary: str) -> str:
    """Apply business rules. If flags detected, escalate to Step Functions for human review.
    Args:
        claim_id: The claim identifier
        verification_summary: Verification decision from pre-processing
        auth_summary: Authentication result
    """
    combined_text = (verification_summary + ' ' + auth_summary).upper()

    # Detect flags via keyword scan
    flags = []
    if "EXCLUSION" in combined_text and "TRIGGERED" in combined_text:
        flags.append("exclusion_triggered")
    if "SUICIDE" in combined_text:
        flags.append("suicide_mentioned")
    if "INCONSISTENC" in combined_text:
        flags.append("cross_document_inconsistency")

    if not flags:
        print(f"   ✅ No flags — auto-approving {claim_id}")
        return json.dumps({
            "claim_id": claim_id,
            "adjudication_decision": "APPROVED",
            "rationale": "All checks passed, no flags detected.",
        })

    # Flags detected — escalate to Step Functions
    print(f"   ⚠️  Flags detected for {claim_id}: {flags}")
    print(f"   → Starting Step Functions execution for human review...")

    claim_summary = (
        f"Claim {claim_id} requires human review.\n"
        f"Flags: {flags}\n"
        f"Auth summary (first 500 chars): {auth_summary[:500]}\n"
        f"Verification summary (first 500 chars): {verification_summary[:500]}"
    )

    sfn_input = json.dumps({
        "claim_id": claim_id,
        "claim_summary": claim_summary,
        "flags": json.dumps(flags),
        "submitted_at": datetime.now(timezone.utc).isoformat() + "Z",
    })

    execution = sfn_client.start_execution(
        stateMachineArn=STATE_MACHINE_ARN,
        name=f"hitl-{claim_id}-{uuid.uuid4().hex[:8]}",
        input=sfn_input,
    )

    EXECUTION_ARN = execution["executionArn"]
    print(f"   ✅ Step Functions execution started")
    print(f"   Execution ARN: {EXECUTION_ARN}")
    print(f"   ⏳ Waiting for human decision...")

    return json.dumps({
        "claim_id": claim_id,
        "adjudication_decision": "ESCALATED",
        "rationale": f"Flags detected: {flags}. Submitted to Step Functions for human review.",
        "execution_arn": EXECUTION_ARN,
    })


# ─────────────────────────────────────────────────────────────────────────────
# Tool 4: Persist claim to DynamoDB
# ─────────────────────────────────────────────────────────────────────────────

@tool
def persist_claim_to_dynamodb(
    claim_id: str, policy_number: str, claimant_name: str,
    date_of_death: str, auth_summary: str, verification_summary: str,
    adjudication_decision: str, adjudication_notes: str,
) -> str:
    """Write claim record to DynamoDB.
    Args:
        claim_id: Claim identifier
        policy_number: Policy number
        claimant_name: Claimant name
        date_of_death: Date of death
        auth_summary: Auth result
        verification_summary: Verification result
        adjudication_decision: APPROVED or ESCALATED
        adjudication_notes: Rationale
    """
    from botocore.exceptions import ClientError
    stage = "pending_human_review" if adjudication_decision == "ESCALATED" else "pipeline_complete"
    record = {
        "claim_id": claim_id, "policy_number": policy_number,
        "claimant_name": claimant_name, "date_of_death": date_of_death,
        "auth_result_summary": auth_summary[:2000],
        "verification_summary": verification_summary[:3000],
        "adjudication_decision": adjudication_decision,
        "adjudication_notes": adjudication_notes,
        "stage": stage,
        "created_at": datetime.now(timezone.utc).isoformat() + "Z",
    }
    try:
        table = dynamodb.create_table(
            TableName=DYNAMODB_TABLE,
            KeySchema=[{"AttributeName": "claim_id", "KeyType": "HASH"}],
            AttributeDefinitions=[{"AttributeName": "claim_id", "AttributeType": "S"}],
            BillingMode="PAY_PER_REQUEST",
        )
        table.wait_until_exists()
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceInUseException":
            table = dynamodb.Table(DYNAMODB_TABLE)
        else: raise
    table.put_item(Item=record)
    return json.dumps({"status": "success", "claim_id": claim_id, "stage": stage})


print("✅ adjudicate_claim and persist_claim_to_dynamodb tools defined")

In [ ]:
SUPERVISOR_SYSTEM_PROMPT = """You are the Intake Orchestrator — the supervisor agent for Insurance Claims Processing.\n\n## Workflow\n1. Call retrieve_claim_documents to download claim documents from S3\n2. Call run_preprocessing_graph — include the EXACT document_paths from retrieve_claim_documents. Do NOT construct filenames yourself.\n3. Call adjudicate_claim with the claim_id, verification_summary, and auth_summary\n4. Call persist_claim_to_dynamodb with ALL results\n5. Summarize what you completed\n\n## Guidelines\n- If adjudication returns ESCALATED, persist with stage pending_human_review and note that human review is required\n- Do NOT send notifications for escalated claims\n- Report what YOU completed — do not echo specialist agent recommendations
"""

supervisor_model = BedrockModel(
    model_id="us.amazon.nova-2-lite-v1:0",
    region_name=REGION,
    additional_request_fields={
        "reasoningConfig": {"type": "enabled", "maxReasoningEffort": "low"}
    },
)

intake_orchestrator = Agent(
    model=supervisor_model,
    system_prompt=SUPERVISOR_SYSTEM_PROMPT,
    tools=[retrieve_claim_documents, run_preprocessing_graph, adjudicate_claim, persist_claim_to_dynamodb],
)

print("✅ Intake Orchestrator defined (4 tools, HITL-aware)")

## Part D — Run the Pipeline (Flagged Claim)

This claim has death circumstances that mention **suicide** — which triggers the `suicide_within_2_years` exclusion in the Socotra policy data. The adjudication tool will detect this flag and escalate to Step Functions instead of auto-approving.

In [ ]:
# Flagged claim: death circumstances trigger suicide exclusion
FLAGGED_CLAIM = {
    "claim_id": "CLM-2026-00201",
    "policy_number": "WL-4582-1093",
    "claimant_name": "Lisa Doe",
    "date_of_death": "2026-01-15",
    "death_circumstances": "Suicide at residence — self-inflicted injury",
}

claim = FLAGGED_CLAIM

supervisor_prompt = (
    "## New Claim Submission\n\n"
    f"**Claim ID:** {claim['claim_id']}\n"
    f"**Policy Number:** {claim['policy_number']}\n"
    f"**Claimant Name:** {claim['claimant_name']}\n"
    f"**Date of Death:** {claim['date_of_death']}\n"
    f"**Death Circumstances:** {claim['death_circumstances']}\n\n"
    f"### S3 Location\nBucket: {S3_BUCKET}\nPrefix: {S3_PREFIX}\n\n"
    "Process this claim through the pipeline: retrieve documents from S3, run pre-processing, "
    "adjudicate, and persist the record."
)

print("🚀 Starting Pipeline (Flagged Claim)")
print("=" * 60)
print(f"   Claim ID     : {claim['claim_id']}")
print(f"   Circumstances: {claim['death_circumstances']}")
print("=" * 60)

with socotra_mcp_client:
    result = intake_orchestrator(supervisor_prompt)

print()
print("=" * 60)
print("✅ Pipeline paused — claim escalated to human review")
print("=" * 60)


## Part E — Act as the Adjudicator

The Step Functions execution is now **paused**, waiting for a human decision. The task token was written to DynamoDB by the state machine.

In this cell, you’ll:
1. Retrieve the pending task from DynamoDB
2. Review the claim summary
3. Enter your decision (approve/deny)
4. Call `send_task_success` to resume the state machine

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Retrieve the pending task and act as the adjudicator
# ─────────────────────────────────────────────────────────────────────────────

claim_id = FLAGGED_CLAIM['claim_id']
table = dynamodb.Table(DYNAMODB_TABLE)

print("⏳ Polling for human review task (this may take a few seconds)...")
resp = sfn_client.get_activity_task(activityArn=ACTIVITY_ARN, workerName="workshop-adjudicator")
task_token = resp.get("taskToken")
task_input = json.loads(resp.get("input", "{}"))

if not task_token:
    print("❌ No task found. The Step Functions execution may not have started yet.")
    print("   Re-run the previous cell and try again.")
else:
    claim_summary = task_input.get("claim_summary", "No summary available")
    flags = task_input.get("flags", "[]")

    print("📋 CLAIM REVIEW")
    print("=" * 60)
    print(f"Claim ID: {claim_id}")
    print(f"Flags: {flags}")
    print()
    print(claim_summary)
    print("=" * 60)
    print()

    # Human decision
    decision = input("🏛️ Your decision (approve/deny): ").strip().upper()
    notes = input("📝 Notes: ").strip()

    # Send the decision back to Step Functions
    sfn_client.send_task_success(
        taskToken=task_token,
        output=json.dumps({
            "decision": decision,
            "notes": notes,
            "adjudicated_at": datetime.now(timezone.utc).isoformat() + "Z",
        })
    )

    # Update DynamoDB with the decision
    table.update_item(
        Key={"claim_id": claim_id},
        UpdateExpression="SET stage = :s, adjudication_decision = :d, adjudication_notes = :n, adjudicated_at = :t",
        ExpressionAttributeValues={
            ":s": "adjudicated",
            ":d": decision,
            ":n": notes,
            ":t": datetime.now(timezone.utc).isoformat() + "Z",
        },
    )

    print()
    print(f"✅ Decision submitted: {decision}")
    print(f"   Notes: {notes}")
    print("   Step Functions execution resumed and DynamoDB updated.")



## Part F — Verify the Loop Closed

After you submitted your decision, Step Functions resumed and updated the DynamoDB record. Let’s verify.

In [ ]:
# Wait a moment for Step Functions to complete
time.sleep(5)

claim_id = FLAGGED_CLAIM['claim_id']
table = dynamodb.Table(DYNAMODB_TABLE)
response = table.get_item(Key={"claim_id": claim_id})
item = response.get("Item", {})

print("📋 Claim Record After Human Review")
print("=" * 60)
print(f"   Claim ID     : {item.get('claim_id')}")
print(f"   Stage        : {item.get('stage')}")
print(f"   Decision     : {item.get('adjudication_decision', 'N/A')}")
print(f"   Notes        : {item.get('adjudication_notes', 'N/A')}")
print(f"   Adjudicated  : {item.get('adjudicated_at', 'N/A')}")
print()
if item.get('stage') == 'adjudicated':
    print("✅ The human-in-the-loop cycle is complete.")
    print("   The claim has been adjudicated and the record updated by Step Functions.")
else:
    print("⏳ Stage is still not 'adjudicated'. Step Functions may still be processing.")
    print("   Wait a few more seconds and re-run this cell.")

## Part G — Production Architecture

In this notebook, you acted as the adjudicator directly in a notebook cell. In production, the human review step would be replaced by a proper UI:

| Workshop (this notebook) | Production |
|---|---|
| `input()` prompt in Jupyter | Web app, email link, or Amazon Quick Automate task queue |
| Task token stored in DynamoDB | Task token routed to a case management system |
| Single reviewer (you) | Role-based routing to claims adjusters |
| Immediate response | Hours or days — Step Functions waits without consuming compute |

The **callback pattern stays the same** regardless of the human interface. The state machine pauses with a task token, and any system that can call `SendTaskSuccess` can resume it.

### Integration with Amazon Quick Automate

In a production deployment, Amazon Quick Automate would:
1. Poll DynamoDB for claims with `stage: pending_human_review`
2. Create cases with the claim summary and flags
3. Route HITL tasks to adjudicators via a form (Approve/Deny dropdown + notes field)
4. On task completion, call `SendTaskSuccess` with the decision
5. Step Functions resumes and updates the claim record

This is an **event-driven handoff** — the Intake Orchestrator writes to DynamoDB and exits. Quick Automate picks up from there. No synchronous coupling between the agent pipeline and the human workflow.

## Part H — Cleanup (Optional)

Delete the Step Functions state machine and IAM role created in this notebook.

In [ ]:
import shutil

# Delete state machine
try:
    sfn_client.delete_state_machine(stateMachineArn=STATE_MACHINE_ARN)
    print(f'✅ Deleted state machine: {STATE_MACHINE_NAME}')
except Exception as e:
    print(f'⚠️  Could not delete state machine: {e}')

# Delete activity
try:
    sfn_client.delete_activity(activityArn=ACTIVITY_ARN)
    print(f'✅ Deleted activity: {ACTIVITY_NAME}')
except Exception as e:
    print(f'⚠️  Could not delete activity: {e}')

# Delete IAM role (must remove inline policy first)
try:
    iam_client.delete_role_policy(RoleName=SFN_ROLE_NAME, PolicyName='claims-hitl-dynamodb-access')
    iam_client.delete_role(RoleName=SFN_ROLE_NAME)
    print(f'✅ Deleted IAM role: {SFN_ROLE_NAME}')
except Exception as e:
    print(f'⚠️  Could not delete IAM role: {e}')

# Remove local folders
for folder in ['claim_documents', 'extracted_output']:
    if Path(folder).exists():
        shutil.rmtree(folder)
        print(f'✅ Removed {folder}/')
    else:
        print(f'ℹ️  {folder}/ not found (already clean)')

